# Hip 라벨 댓글 크롤링 (코르티스 / 올데이프로젝트)

**목적:** `purpose = train` — RoBERTa 학습용  
**목표:** 아티스트당 1만 개 이상, 총 2~3만 개  
**저장 경로:** Google Drive `/FNC/data/train/hip/`

### 실행 전 필요한 것
1. **YouTube Data API v3 키** — [Google Cloud Console](https://console.cloud.google.com/) → API 및 서비스 → 사용자 인증 정보 → API 키 생성
2. **채널 ID** — 아티스트 YouTube 채널 페이지 접속 → URL에서 `/channel/UC...` 부분 복사  
   (또는 채널 About → 공유 → 채널 ID 복사)

video_id를 하나씩 입력할 필요 없이, **채널 ID 하나**로 해당 채널의 영상을 자동 탐색합니다.

## 0. Colab 환경 설정

In [ ]:
# Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q google-api-python-client

## 1. 설정

In [ ]:
import json
import hashlib
import html
import re
import time
from collections import Counter
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from tqdm.notebook import tqdm

from google.colab import userdata
API_KEY = userdata.get("YOUTUBE_API_KEY")

DRIVE_ROOT = Path("/content/drive/MyDrive/FNC")
SAVE_DIR   = DRIVE_ROOT / "data/train/hip"
QUOTA_FILE = DRIVE_ROOT / "crawler/quota_usage.json"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

youtube = build("youtube", "v3", developerKey=API_KEY)
print("설정 완료")

설정 완료


## 2. 채널 설정

채널 ID 찾는 법:
- YouTube 채널 페이지 URL이 `youtube.com/channel/UCxxxx` 형태이면 그대로 복사
- `youtube.com/@코르티스` 형태이면: 채널 About 탭 → 공유 → 채널 ID 복사

@cortis_bighit

UCZMYvSPulDSUdx7bdtTFdrg

@ALLDAY_PROJECT

UCKOY4n0sv1pAPs5E4LxtSXQ

In [ ]:
CHANNELS = [
    {
        "artist": "코르티스",
        "channel_id": "UCZMYvSPulDSUdx7bdtTFdrg",   # ← UC로 시작하는 24자리
        "label": "hip",
        "purpose": "train",
    },
    {
        "artist": "올데이프로젝트",
        "channel_id": "UCKOY4n0sv1pAPs5E4LxtSXQ",
        "label": "hip",
        "purpose": "train",
    },
]

# 영상 타입 분류 키워드 (제목 기반)
VIDEO_TYPE_RULES = [
    ("teaser",  ["티저", "teaser", "tease"]),
    ("fancam",  ["직캠", "fancam", "fan cam"]),
    ("behind",  ["비하인드", "behind", "making"]),
    ("MV",      ["mv", "music video", "뮤직비디오", "official"]),
]

def classify_video_type(title: str) -> str:
    t = title.lower()
    for vtype, keywords in VIDEO_TYPE_RULES:
        if any(k in t for k in keywords):
            return vtype
    return "other"

## 3. 할당량 관리자

| API 호출 | 비용 |
|---|---|
| `search.list` (영상 탐색) | 100 units/call |
| `commentThreads.list` (댓글 100개) | 1 unit/call |
| `videos.list` (영상 메타데이터) | 1 unit/call |

무료 한도 **10,000 units/day** — 영상 탐색에 최대 400 units(채널당 2 calls) 사용, 나머지 9,600 units로 댓글 수집.

In [ ]:
from datetime import timedelta

# YouTube API 할당량은 미국 태평양 시간(PT) 자정에 리셋됨 — UTC 기준으로 추적하면 최대 8시간 오차 발생
_PT = timezone(timedelta(hours=-8))  # PST 고정 (PDT와 1시간 차이는 무시 가능)

class QuotaManager:
    DAILY_LIMIT = 10_000

    def __init__(self, path: Path):
        self.path = path
        self._data = json.loads(path.read_text()) if path.exists() else {}
        self._pending = 0  # 미저장 누적 — Drive I/O 최소화

    def _today(self) -> str:
        return datetime.now(_PT).strftime("%Y-%m-%d")  # PT 기준 날짜

    def used_today(self) -> int:
        return self._data.get(self._today(), 0) + self._pending

    def remaining(self) -> int:
        return self.DAILY_LIMIT - self.used_today()

    def consume(self, units: int = 1):
        self._pending += units
        if self._pending >= 100:  # 100 units 쌓일 때만 Drive에 기록
            self.flush()
        if self.used_today() > self.DAILY_LIMIT:
            raise RuntimeError(f"일일 할당량 초과: {self.used_today():,} units")

    def flush(self):
        """남은 누적 사용량을 Drive 파일에 저장. 수집 완료 후 반드시 호출."""
        day = self._today()
        self._data[day] = self._data.get(day, 0) + self._pending
        self._pending = 0
        self.path.parent.mkdir(parents=True, exist_ok=True)
        self.path.write_text(json.dumps(self._data, indent=2))

    def status(self):
        print(f"오늘 사용: {self.used_today():,} / {self.DAILY_LIMIT:,} units  |  남은: {self.remaining():,} units")


quota = QuotaManager(QUOTA_FILE)
quota.status()

오늘 사용: 0 / 10,000 units  |  남은: 10,000 units


## 4. 채널 → 영상 목록 자동 탐색

`search.list`로 채널 내 영상을 최대 50개씩 페이지 탐색합니다.  
100 units/call이므로 채널당 2~3 calls(200~300 units)로 수십 개 영상 ID를 한 번에 확보합니다.

In [ ]:
def fetch_channel_videos(channel_id: str, max_results: int = 100) -> list[dict]:
    """채널의 영상 목록을 자동 탐색. search.list 100 units/call."""
    videos = []
    page_token = None

    while len(videos) < max_results:
        resp = youtube.search().list(
            part="snippet",
            channelId=channel_id,
            type="video",
            order="viewCount",      # 조회수 높은 영상 우선 → 댓글도 많음
            maxResults=50,
            pageToken=page_token,
        ).execute()
        quota.consume(100)

        for item in resp.get("items", []):
            vid = item["id"]["videoId"]
            title = item["snippet"]["title"]
            published_at = item["snippet"]["publishedAt"]
            videos.append({
                "video_id": vid,
                "title": title,
                "video_published_at": published_at,
                "video_type": classify_video_type(title),
            })

        page_token = resp.get("nextPageToken")
        if not page_token:
            break
        time.sleep(0.5)

    return videos


# 채널별 영상 목록 탐색
channel_videos = {}   # { artist: [video_dict, ...] }

for ch in CHANNELS:
    if "여기에" in ch["channel_id"]:
        print(f"⏭  {ch['artist']}: 채널 ID 미입력, 건너뜀")
        continue
    print(f"\n▶ {ch['artist']} 채널 영상 탐색 중...")
    videos = fetch_channel_videos(ch["channel_id"], max_results=100)
    channel_videos[ch["artist"]] = videos
    print(f"  발견된 영상: {len(videos)}개")
    quota.status()


▶ 코르티스 채널 영상 탐색 중...
  발견된 영상: 100개
오늘 사용: 200 / 10,000 units  |  남은: 9,800 units

▶ 올데이프로젝트 채널 영상 탐색 중...
  발견된 영상: 100개
오늘 사용: 400 / 10,000 units  |  남은: 9,600 units


## 4-2. 소속사/타 채널 영상 보완 탐색 (검색 기반)

채널 탐색만으로는 소속사 공식 채널에 올라간 MV·수록곡이 빠질 수 있습니다.  
아티스트명 키워드 검색으로 전체 YouTube를 대상으로 탐색해서 채널 결과에 병합합니다.  
비용: 쿼리당 100 units (아티스트당 3쿼리 × 2명 = **600 units** 추가)

In [8]:
# 아티스트별 검색 쿼리 — 한국어/영어 명칭 모두 포함해서 소속사 채널 MV·수록곡 탐색
SEARCH_QUERIES = {
    "코르티스":     ["CORTIS MV", "코르티스 MV", "CORTIS 코르티스 live performance"],
    "올데이프로젝트": ["ALLDAY PROJECT MV", "올데이프로젝트 MV", "ALLDAY PROJECT 뮤직비디오"],
}

def search_videos_by_query(query: str, max_results: int = 50) -> list[dict]:
    """키워드 검색 — 채널 무관하게 전체 YouTube 탐색. 100 units/call."""
    videos, page_token = [], None
    while len(videos) < max_results:
        if quota.remaining() < 200:
            print("⚠️  할당량 부족으로 검색 중단")
            break
        try:
            resp = youtube.search().list(
                part="snippet",
                q=query,
                type="video",
                order="viewCount",   # 조회수 높은 순 → 댓글 많은 MV 우선
                maxResults=50,
                pageToken=page_token,
            ).execute()
        except HttpError as e:
            print(f"  검색 오류 {e.resp.status}: {query}")
            break
        quota.consume(100)
        for item in resp.get("items", []):
            videos.append({
                "video_id":           item["id"]["videoId"],
                "title":              item["snippet"]["title"],
                "video_published_at": item["snippet"]["publishedAt"],
                "video_type":         classify_video_type(item["snippet"]["title"]),
            })
        page_token = resp.get("nextPageToken")
        if not page_token:
            break
        time.sleep(0.5)
    return videos


# 채널 탐색 결과에 검색 결과 병합 (video_id 기준 중복 제거)
for ch in CHANNELS:
    artist = ch["artist"]
    if artist not in channel_videos:
        continue

    existing_ids = {v["video_id"] for v in channel_videos[artist]}
    added = 0

    for query in SEARCH_QUERIES.get(artist, []):
        print(f"  [{artist}] '{query}' 검색 중...")
        for v in search_videos_by_query(query):
            if v["video_id"] not in existing_ids:
                channel_videos[artist].append(v)
                existing_ids.add(v["video_id"])
                added += 1
        time.sleep(0.5)

    print(f"  → {artist}: {added}개 추가 (소속사 채널 포함) / 총 {len(channel_videos[artist])}개")

quota.status()

  [코르티스] 'CORTIS MV' 검색 중...
  [코르티스] '코르티스 MV' 검색 중...
  [코르티스] 'CORTIS 코르티스 live performance' 검색 중...
  → 코르티스: 59개 추가 (소속사 채널 포함) / 총 159개
  [올데이프로젝트] 'ALLDAY PROJECT MV' 검색 중...
  [올데이프로젝트] '올데이프로젝트 MV' 검색 중...
  [올데이프로젝트] 'ALLDAY PROJECT 뮤직비디오' 검색 중...
  → 올데이프로젝트: 60개 추가 (소속사 채널 포함) / 총 160개
오늘 사용: 1,000 / 10,000 units  |  남은: 9,000 units


## 5. 수집 대상 영상 확인 및 필터링

자동 탐색된 영상 목록을 확인하고 필요 시 제외할 영상을 지정합니다.

In [9]:
# 영상 목록 확인
for artist, videos in channel_videos.items():
    print(f"\n{'='*55}")
    print(f"[{artist}] 총 {len(videos)}개 영상")
    print(f"{'='*55}")
    for i, v in enumerate(videos):
        print(f"  [{i:2d}] {v['video_type']:8s} | {v['video_published_at'][:10]} | {v['title'][:50]}")


[코르티스] 총 159개 영상
  [ 0] other    | 2025-10-26 | taking FaSHioN to the next level w/ #KATSEYE #KATS
  [ 1] other    | 2025-08-22 | CORTIS (코르티스) ‘GO!’ Dance Practice (Fix ver.)
  [ 2] other    | 2026-04-28 | #KATSEYE_Yoonchae 선배님, #KATSEYE_Daniela 선배님과 that&
  [ 3] other    | 2025-09-14 | CORTIS (코르티스) ‘FaSHioN&#39; Dance Practice (Fix ve
  [ 4] other    | 2025-09-30 | fire remix we found🔥 #CORTIS #코르티스 #CORTIS_FaSHioN
  [ 5] other    | 2025-08-22 | Pack Up Bro
  [ 6] other    | 2025-08-26 | #바다 선배님과 함께 GO? Let&#39;s go!!!🔥 #BadaLee #CORTIS 
  [ 7] other    | 2025-10-03 | CORTIS (코르티스) &#39;GO!&#39; Live Performance @ REL
  [ 8] other    | 2026-04-25 | we&#39;re cortis who love #jhope 선배님감사합니다 선배님 ❤️‍🔥
  [ 9] other    | 2025-09-19 | poses to try #CORTIS #코르티스
  [10] other    | 2025-11-07 | We&#39;re on a JoyRide
  [11] other    | 2025-08-24 | #KIMCHAEWON 선배님 #KAZUHA 선배님 #HONGEUNCHAE 선배님과 Diff
  [12] other    | 2025-11-06 | megaknighted by Martin 🫨 #CORTIS #코르티스
  [13] other    | 2025-1

In [10]:
# 제외할 영상 인덱스 지정 (위 출력의 [번호] 참고)
# 팬이 올린 커버, 관계없는 영상 등을 제외
EXCLUDE = {
      "코르티스": [
          85,   # Kawasaki/Santos Bravos — 무관
          100,  # SEVENTEEN 'HOT' MV
          101,  # LE SSERAFIM 'SPAGHETTI' MV
          102,  # TXT 'Deja Vu' MV
          103,  # Hearts2Hearts 'RUDE!' MV
          106,  # ILLIT 'It's Me' MV
          107,  # ENHYPEN 'Knife' MV
          111,  # K-POP Charts News Live 채널
          113,  # TWS 'OVERDRIVE' MV
          115,  # &TEAM 'Lunatic' MV
          117,  # UNCHILD MV
      ],
      "올데이프로젝트": [
          100,  # Now United - All Day (완전 다른 아티스트)
          101,  # CORTIS FaSHioN MV (잘못 섞임)
          153,  # K-POP Charts News Live 채널
          154,  # BE:FIRST ALL DAY MV (일본 아티스트)
      ],
}


# 필터 적용
VIDEO_TARGETS = []
for ch in CHANNELS:
    artist = ch["artist"]
    if artist not in channel_videos:
        continue
    exclude_idx = set(EXCLUDE.get(artist, []))
    for i, v in enumerate(channel_videos[artist]):
        if i in exclude_idx:
            continue
        VIDEO_TARGETS.append({
            **v,
            "artist": artist,
            "label": ch["label"],
            "purpose": ch["purpose"],
        })

print(f"최종 수집 대상: {len(VIDEO_TARGETS)}개 영상")

# 타입별 분포 확인
from collections import Counter
type_counts = Counter(v["video_type"] for v in VIDEO_TARGETS)
for vtype, cnt in type_counts.most_common():
    print(f"  {vtype:10s}: {cnt}개")

최종 수집 대상: 304개 영상
  other     : 285개
  MV        : 15개
  behind    : 3개
  fancam    : 1개


## 6. 댓글 수집 함수

In [11]:
def crawl_comments(target: dict, max_comments: int = 10_000) -> list[dict]:
    collected, page_token = [], None
    video_id = target["video_id"]

    pbar = tqdm(
        total=max_comments,
        desc=f"{target['artist']} | {target['video_type']} | {target['title'][:30]}",
        leave=False,
    )

    while len(collected) < max_comments:
        if quota.remaining() < 50:
            print("⚠️  할당량 소진 임박. 수집 중단.")
            break

        try:
            resp = youtube.commentThreads().list(
                part="snippet",
                videoId=video_id,
                maxResults=100,
                pageToken=page_token,
                order="time",
                textFormat="plainText",
            ).execute()
        except HttpError as e:
            if e.resp.status == 403:
                try:
                    reason = json.loads(e.content)["error"]["errors"][0]["reason"]
                except Exception:
                    reason = "unknown"
                if reason in ("quotaExceeded", "rateLimitExceeded"):
                    print(f"  🚫  API 할당량 초과 ({reason}). 전체 수집 중단.")
                    quota.flush()
                    raise
                elif reason == "commentsDisabled":
                    print(f"  ⚠️  댓글 비활성화: {target['title'][:40]}")
                else:
                    print(f"  ⚠️  403 오류 ({reason}): {target['title'][:40]}")
            elif e.resp.status == 404:
                print(f"  ⚠️  영상 없음 (삭제/비공개): {target['title'][:40]}")
            else:
                print(f"  API 오류 {e.resp.status}: {target['title'][:40]}")
            break

        quota.consume(1)
        crawled_at = datetime.now(timezone.utc).isoformat()

        for item in resp.get("items", []):
            snip = item["snippet"]["topLevelComment"]["snippet"]
            collected.append({
                "video_id":           video_id,
                "video_type":         target["video_type"],
                "video_title":        target["title"],       # monstax.jsonl 스키마 맞춤
                "video_published_at": target.get("video_published_at", ""),
                "artist":             target["artist"],
                "label":              target["label"],
                "purpose":            target["purpose"],
                "comment_id":         item["snippet"]["topLevelComment"]["id"],
                "text":               snip["textDisplay"],
                "likes":              snip["likeCount"],
                "published_at":       snip["publishedAt"],
                "crawled_at":         crawled_at,            # monstax.jsonl 스키마 맞춤
            })
            pbar.update(1)

        page_token = resp.get("nextPageToken")
        if not page_token:
            break

        time.sleep(0.2)

    pbar.close()
    return collected

## 7. 전처리 함수

In [12]:
import html as html_module

HTML_TAG_RE = re.compile(r"<[^>]+>")
SPAM_RE     = re.compile(r"http[s]?://|www\.|\.com|\.kr")
REPEAT_RE   = re.compile(r"(.)\1{4,}")
HANGUL_RE   = re.compile(r"[가-힣]")

def has_enough_korean(text: str, min_ratio: float = 0.2) -> bool:
    """한글 비율이 min_ratio 이상이면 한국어로 판단.
    langdetect는 짧은 팬 댓글/이모지 혼용 텍스트에서 오판이 잦아서 대체.
    """
    if not text:
        return False
    return len(HANGUL_RE.findall(text)) / len(text) >= min_ratio

def clean(records: list[dict]) -> list[dict]:
    seen, cleaned = set(), []
    for r in records:
        text = r["text"].strip()

        # 1. HTML 태그/엔티티 제거 (textDisplay에 <br>, <a href> 포함될 수 있음)
        text = HTML_TAG_RE.sub(" ", text)
        text = html_module.unescape(text)
        text = re.sub(r" {2,}", " ", text).strip()

        # 2. 최소 길이 — "개멋있어", "역시 밴드" 같은 핵심 라벨 문장 보존
        if len(text) < 4:
            continue

        # 3. 중복 제거
        h = hashlib.md5(text.encode()).hexdigest()
        if h in seen:
            continue
        seen.add(h)

        # 4. 스팸/URL 필터
        if SPAM_RE.search(text) or REPEAT_RE.search(text):
            continue

        # 5. 한글 비율 필터
        if not has_enough_korean(text):
            continue

        r["text"] = text
        cleaned.append(r)
    return cleaned

## 8. 전체 실행

In [13]:
TARGET_PER_ARTIST = 10_000  # 아티스트당 최소 목표 수량
MAX_PER_VIDEO     = 10_000  # 영어 댓글 필터링 손실 고려해서 넉넉하게

all_clean    = []
artist_stats = {}

try:
    for target in VIDEO_TARGETS:
        raw     = crawl_comments(target, max_comments=MAX_PER_VIDEO)
        cleaned = clean(raw)

        artist = target["artist"]
        artist_stats.setdefault(artist, {"raw": 0, "clean": 0})
        artist_stats[artist]["raw"]   += len(raw)
        artist_stats[artist]["clean"] += len(cleaned)
        all_clean.extend(cleaned)
        time.sleep(0.5)
except HttpError:
    print("할당량 초과로 수집 중단. 지금까지 수집된 데이터를 저장합니다.")
finally:
    quota.flush()

print("\n[수집 결과]")
for artist, s in artist_stats.items():
    status = "✅" if s["clean"] >= TARGET_PER_ARTIST else f"⚠️  목표 미달 ({TARGET_PER_ARTIST:,}개 필요)"
    print(f"  {artist}: {s['raw']:,}개 수집 → {s['clean']:,}개 (전처리 후)  {status}")
print(f"  합계: {len(all_clean):,}개")
quota.status()

코르티스 | other | taking FaSHioN to the next lev:   0%|          | 0/10000 [00:00<?, ?it/s]

코르티스 | other | CORTIS (코르티스) ‘GO!’ Dance Prac:   0%|          | 0/10000 [00:00<?, ?it/s]

코르티스 | other | #KATSEYE_Yoonchae 선배님, #KATSEY:   0%|          | 0/10000 [00:00<?, ?it/s]

코르티스 | other | CORTIS (코르티스) ‘FaSHioN&#39; Da:   0%|          | 0/10000 [00:00<?, ?it/s]

코르티스 | other | fire remix we found🔥 #CORTIS #:   0%|          | 0/10000 [00:00<?, ?it/s]

...

올데이프로젝트 | other | 알고보면 더 소름돋는 아이돌 뮤비 촬영의 비밀 TOP4:   0%|          | 0/10000 [00:00<?, ?it/s]

올데이프로젝트 | other | &#39;FAMOUS&#39; 뮤비를 위해 냅다 무릎 :   0%|          | 0/10000 [00:00<?, ?it/s]

올데이프로젝트 | MV | [배틀그라운드] PUBG x ALLDAY PROJECT:   0%|          | 0/10000 [00:00<?, ?it/s]

올데이프로젝트 | other | 🐘 🐘 🐘 #MMA2025 #ALLDAYPROJECT :   0%|          | 0/10000 [00:00<?, ?it/s]


[수집 결과]
  코르티스: 457,335개 수집 → 36,513개 (전처리 후)  ✅
  올데이프로젝트: 193,477개 수집 → 76,789개 (전처리 후)  ✅
  합계: 113,302개
오늘 사용: 7,683 / 10,000 units  |  남은: 2,317 units


## 9. Google Drive 저장

In [14]:
# 글로벌 중복 제거 — 영상 간, 아티스트 간 동일 댓글 제거
# clean() 내부의 seen은 호출마다 리셋되므로 여기서 전체 범위 dedupe 필요
seen_global, deduped = set(), []
for r in all_clean:
    # comment_id 우선, 없으면 텍스트 해시로 fallback
    key = r.get("comment_id") or hashlib.md5(r["text"].encode()).hexdigest()
    if key not in seen_global:
        seen_global.add(key)
        deduped.append(r)

removed  = len(all_clean) - len(deduped)
all_clean = deduped
print(f"글로벌 중복 제거: {removed:,}개 제거 → 최종 {len(all_clean):,}개")

글로벌 중복 제거: 0개 제거 → 최종 113,302개


In [15]:
if not all_clean:
    print("⚠️  수집된 데이터 없음. 저장 건너뜀.")
else:
    df = pd.DataFrame(all_clean)

    for artist, group in df.groupby("artist"):
        fname = SAVE_DIR / f"{artist}.jsonl"
        group.to_json(fname, orient="records", lines=True, force_ascii=False)
        print(f"저장 완료: {fname}  ({len(group):,}개)")

    all_fname = SAVE_DIR / "hip_all.jsonl"
    df.to_json(all_fname, orient="records", lines=True, force_ascii=False)
    print(f"저장 완료: {all_fname}  (총 {len(df):,}개)")

저장 완료: /content/drive/MyDrive/FNC/data/train/hip/올데이프로젝트.jsonl  (76,789개)
저장 완료: /content/drive/MyDrive/FNC/data/train/hip/코르티스.jsonl  (36,513개)
저장 완료: /content/drive/MyDrive/FNC/data/train/hip/hip_all.jsonl  (총 113,302개)


## 10. 검증

In [16]:
if not all_clean:
    print("데이터 없음")
else:
    print("아티스트별")
    print(df.groupby("artist")["text"].count().to_string())

    print("\n영상 타입별")
    print(df.groupby("video_type")["text"].count().to_string())

    print("\npublished_at 형식 확인 (ISO 8601 UTC)")
    for s in df["published_at"].head(3):
        assert "T" in s and "Z" in s, f"형식 오류: {s}"
        print(f"  OK: {s}")

    print("\n샘플 댓글 5개")
    for t in df["text"].sample(min(5, len(df)), random_state=42):
        print(f"  • {t}")

    print()
    quota.status()

아티스트별
artist
올데이프로젝트    76789
코르티스       36513

영상 타입별
video_type
MV          4015
behind      1600
fancam      2169
other     105518

published_at 형식 확인 (ISO 8601 UTC)
  OK: 2026-05-16T12:18:44Z
  OK: 2026-05-11T16:32:17Z
  OK: 2026-04-18T11:10:44Z

샘플 댓글 5개
  • 콘서트가 기대되는 그룹 1위!!!
애들아 우린 Moshpit 할 준비가 되어 있단다👽 우리 내년에 꼭 콘서트 아니면 팬 미팅 하자!!!
  • 아니 영서언니 다 이쁜데 이마가 특히 이쁨
  • 아 엄나 좋아
  • 마틴❤❤❤
🎉🎉🎉
  • 기괴하다

오늘 사용: 7,683 / 10,000 units  |  남은: 2,317 units
